In [17]:
%pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [18]:
%pip install pandas
%pip install python-dotenv
%pip install supabase

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [19]:
from dotenv import load_dotenv
import os
from supabase import create_client, Client
import pandas as pd

In [20]:
load_dotenv(dotenv_path="/Users/charl/OneDrive/Documents/GitHub/IPFlow/.env")

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

In [21]:
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

In [22]:
df = pd.DataFrame(
    supabase.table("patents").select("*").execute().data
)
df

,patent_id,title,status,expiry_date,filing_date,abstract,inventor_name,raw_xml,publication_date
0,40,IMPROVEMENTS FOR CONNECTING TOGETHER THE ENDS ...,Expired,None,1867-04-29,The object of the improvements is to obtain a ...,HENRY LAMPSON,"<PatentInformationResponse xmlns=""http://www.i...",1873-12-31
1,205,IMPROVEMENTS IN THE MANUFACTURE OF COOKING RANGES,Expired,None,1873-12-01,This cooking range has been named the. New. Ze...,HARRY DANIEL MANNING,"<PatentInformationResponse xmlns=""http://www.i...",1873-12-31
2,206,INVENTION FOR A FORM OF TURBINE WHEREBY FOOTST...,Expired,None,1873-12-04,Invention for a. Form of. Turbine whereby. Foo...,GEORGE BLACK,"<PatentInformationResponse xmlns=""http://www.i...",1873-12-31
3,207,AN IMPROVED PROCESS OF AND MACHINERY FOR TREAT...,Expired,None,1873-12-12,An. Improved. Process of and. Machinery for. T...,JAMES HILL DICKSON,"<PatentInformationResponse xmlns=""http://www.i...",1873-12-31
4,1,AN INVENTION FOR THE PREPARATION OF THE FIBRE ...,Expired,None,1861-03-26,The specific object of the invention is to sep...,ARTHUR GUYON PURCHAS,"<PatentInformationResponse xmlns=""http://www.i...",1873-12-31
...,...,...,...,...,...,...,...,...,...
628,786148,KV1.3 BLOCKERS,Granted,2040-09-18,2020-09-18,The present invention provides novel blockers ...,ZEALAND PHARMA A/S,"<PatentInformationResponse xmlns=""http://www.i...",2025-09-26
629,786149,SOLID FOOD AND SOLID MILK,Under Examination,2040-09-03,2020-09-03,Provided are a solid food and solid milk havin...,"Meiji Co., Ltd.","<PatentInformationResponse xmlns=""http://www.i...",None
630,786150,SOLID FOOD AND SOLID MILK,Under Examination,2040-09-03,2020-09-03,Provided are a solid food that enables the man...,"Meiji Co., Ltd.","<PatentInformationResponse xmlns=""http://www.i...",None
631,786300,"FUSED PYRIDONE COMPOUND, AND PREPARATION METHO...",Filed,2040-09-21,2020-09-21,Disclosed in the present invention are a fused...,"SHANGHAI JEYOU PHARMACEUTICAL CO., LTD","<PatentInformationResponse xmlns=""http://www.i...",None


In [29]:
import re

industry_keywords = {
    "Pharmaceuticals/Biotech": [
        "drug", "pharmaceutical", "antibody", "vaccine", "therapeutic",
        "protein", "gene therapy", "biomarker", "clinical", "dosage form",
        "compound", "formulation", "cancer treatment", "peptide", 
        "biologic", "small molecule", "pharmacology", "pharmacokinetics",
        "pharmacodynamics", "pharmaceutical composition", "biopharmaceutical",
        "biotechnology", "genetic engineering", "molecular biology", "molecular",
        "cell therapy", "immunotherapy", "drug delivery system", "pharmaceutical product",
        "pharmaceutical formulation", "pharmaceutical composition", "pharmaceutical preparation",
        "pharmaceutical agent", "pharmaceutical compound", "pharmaceutical substance",
        "pharmaceutical dosage form", "pharmaceutical excipient", "pharmaceutical additive",
        "pharmaceutical carrier", "pharmaceutical stabilizer", "pharmaceutical preservative",
        "gene editing", "genome editing", "CRISPR", "RNA interference", "siRNA", "antisense oligonucleotide",
        "monoclonal antibody", "polyclonal antibody", "antibody-drug conjugate", "immunoconjugate", 
        "immunotoxin", "immunomodulator", "immunosuppressant",
        "immunostimulant", "immunoadjuvant", "immunogenic", "immunogenicity", "immunotherapy agent", 
        "immunotherapy composition", "immunotherapy formulation",
        "immunotherapy preparation", "immunotherapy product", "immunotherapy substance",
        "peptide therapeutic", "peptide drug", "peptide formulation", "peptide composition",
        "peptide preparation", "peptide product", "peptide substance", "peptide excipient", 
        "peptide additive", "peptide carrier", "peptide stabilizer",
    ],
    "Medical Devices": [
        "implant", "catheter", "prosthetic", "surgical instrument",
        "diagnostic device", "imaging system", "stent", "biosensor",
        "wearable device", "medical device", "set", "syringe", "endoscope", "pacemaker",
        "defibrillator", "ultrasound device", "electrocardiogram", "ECG",
        "MRI", "CT scanner", "X-ray machine", "ventilator", "dialysis machine",
        "hearing aid", "insulin pump", "blood glucose monitor", "infusion pump",
        "respiratory device", "neurostimulator", "orthopedic implant", "dental implant",
        "ophthalmic device", "laser surgery device", "surgical robot", "biomedical sensor", 
        "telemedicine device", "rehabilitation device", "prosthetic limb",
        "orthopedic brace", "cardiac monitor", "endoscopic camera", "surgical navigation system", 
        "biopsy device", "wound care device", "medical imaging software", "patient monitoring system", 
        "surgical stapler", "electrosurgical device", "blood pressure monitor", "pulse oximeter", 
        "defibrillation device", "respiratory ventilator", "dialysis machine", "hemodialysis device", 
        "peritoneal dialysis device", "urinary catheter", "urinary incontinence device", 
        "urological device", "gastrointestinal device", "cardiovascular device", "urology", 
        "orthopedics", "ophthalmology", "neurology", "oncology", "radiology",
        "surgery", "anesthesia", "emergency medicine", "critical care",
    ],
    "Software/IT": [
        "algorithm", "software", "machine learning", "neural network",
        "database", "cloud computing", "user interface", "data processing",
        "artificial intelligence", "computer program", "server", "application",
        "network security", "data analytics", "AI", "Artificial Intelligence", 
        "Deep Learning", "Natural Language Processing", "NLP",
        "Computer Vision", "CV", "Data Mining", "Big Data", "Data Science",
        "Cybersecurity", "Blockchain", "IoT", "Internet of Things",
        "Augmented Reality", "AR", "Virtual Reality", "VR", "Robotics",
        "Software Development", "Software Engineering", "DevOps", "Agile",
        "Cloud Services", "Cloud Infrastructure", "Cloud Platform", "Cloud Application",
        "Mobile Application", "Web Application", "Web Development", "Frontend Development",
        "Backend Development", "Full Stack Development", "Database Management", "Data Storage",
        "Data Retrieval", "Data Visualization", "Data Analysis", "Data Processing",
        "Data Integration", "Data Transformation", "Data Cleaning", "Data Preprocessing",
        "Data Modeling", "Data Warehousing", "Data Governance", "Data Security",
        "Data Privacy", "Data Compliance", "Data Encryption", "Data Backup", "Data Recovery",
        "Data Migration", "Data Synchronization", "Data Replication", "Data Archiving",
        "Data Streaming", "Data Pipeline", "Data Orchestration", "Data Monitoring",
        "Data Logging", "Data Auditing", "Data Reporting", "Data Dashboard",
        "Software Testing", "Software Quality Assurance", "Software Deployment", "Software Maintenance",
        "Software Version Control", "Software Configuration Management", "Software Release Management",
        "Software Documentation", "Software Localization", "Software Internationalization",
        "Software Accessibility", "Software Usability", "Software Performance Optimization",
        "Software Scalability", "Software Reliability", "Software Availability", "Software Fault Tolerance",
        "streaming", "robotics", "robot", "internet", "cloud", "data", "analytics", "cybersecurity", 
        "blockchain", "IoT", "augmented reality"
    ],
    "Semiconductors/Electronics": [
        "semiconductor", "transistor", "circuit", "microprocessor",
        "integrated circuit", "chip", "wafer", "diode", "capacitor",
        "resistor", "electronic device", "optoelectronics", "photodiode",
        "LED", "photovoltaic cell", "MEMS", "ASIC", "FPGA", "analog circuit",
        "digital circuit", "signal processing", "power electronics", "RF circuit",
        "sensor", "microcontroller", "PCB", "printed circuit board", "IC design",
        "semiconductor fabrication", "semiconductor manufacturing", "semiconductor device",
        "semiconductor process", "semiconductor material", "semiconductor technology",
        "semiconductor industry", "semiconductor equipment", "semiconductor testing",
        "semiconductor packaging", "semiconductor assembly", "semiconductor reliability",
        "semiconductor characterization", "semiconductor modeling", "semiconductor simulation",
        "semiconductor physics", "semiconductor chemistry", "semiconductor engineering",
        "circuit design", "circuit simulation", "circuit analysis", "circuit optimization",
        "circuit testing", "circuit prototyping", "circuit fabrication", "circuit layout", "circuit board design", 
        "circuit board fabrication", "circuit board assembly",
        "circuit board testing", "circuit board repair", "circuit board troubleshooting",
        "microprocessor design", "microprocessor architecture", "microprocessor programming",
        "microprocessor testing", "microprocessor fabrication", "microprocessor manufacturing",
        "microprocessor performance", "microprocessor power consumption", "microprocessor heat dissipation",
        "capacitor design", "capacitor fabrication", "capacitor testing", "capacitor performance",
        "capacitor reliability", "capacitor failure analysis", "capacitor material", "capacitor dielectric",
          "capacitor energy storage", "capacitor voltage rating", "capacitor capacitance value",
        "resistor design", "resistor fabrication", "resistor testing", "resistor performance", 
        "resistor reliability", "resistor failure analysis", "resistor material", "resistor resistance value", 
        "resistor power rating"
    ],
    "Telecommunications": [
        "wireless", "antenna", "network protocol", "signal transmission",
        "base station", "bandwidth", "5g", "4g", "3g", "modulation", "telecommunication", 
        "cellular", "satellite communication", "fiber optics", "wire line communication", 
        "telecom", "communication system", "data transmission", "bandwidth allocation", "frequency spectrum", "network architecture",
        "network topology", "network security", "network management", "network optimization", "network",
        "telecommunication system", "telecommunication network", "telecommunication protocol",
        "telecommunication service", "telecommunication infrastructure", "telecommunication equipment",
        "telecommunication technology", "telecommunication industry", "telecommunication regulation",
        "telecommunication standard", "telecommunication policy", "telecommunication research",
        "telecommunication development", "telecommunication innovation", "telecommunication application",
        "telecommunication integration", "telecommunication interoperability", "telecommunication performance",
        "cellular network", "cellular communication", "cellular technology", "cellular service",
        "cellular infrastructure", "cellular equipment", "cellular standard", "cellular protocol", 
        "cellular frequency", "cellular bandwidth", "cellular modulation",
        "cellular signal", "cellular coverage", "cellular capacity", "cellular optimization",
        "cellular management", "cellular security", "cellular interoperability", 
        "satellite network", "satellite technology", "satellite service", 
        "satellite infrastructure", "satellite equipment", "satellite standard", "satellite protocol",
        "satellite frequency", "satellite bandwidth", "satellite modulation",

    ],
    "Automotive": [
        "vehicle", "engine", "automobile", "transmission system",
        "braking system", "autonomous driving", "chassis", "fuel injection", 
        "spark plug", "tire", "oil filter", "brake pad", "exhaust system", "suspension", 
        "steering system", "ignition system", "cooling system", "airbag", "seat belt", "headlight",
        "taillight", "windshield", "rearview mirror", "dashboard", "speedometer",
        "odometer", "tachometer", "fuel gauge", "temperature gauge", "oil pressure gauge",
        "battery", "alternator", "starter motor", "radiator", "fan belt",
        "timing belt", "timing chain", "camshaft", "crankshaft", "piston", "cylinder",
        "valve", "spark plug wire", "ignition coil", "fuel pump",
        "fuel tank", "fuel line", "fuel filter", "air filter", "oil filter",
        "brake disc", "brake drum", "brake caliper", "brake rotor",
        "brake master cylinder", "brake booster", "brake fluid", "brake line",
        "brake pad", "brake shoe", "brake pedal", "clutch", "clutch disc",
        "clutch pressure plate", "clutch master cylinder", "clutch slave cylinder",
        "clutch cable", "clutch fork", "clutch release bearing", "clutch hydraulic line",
        "clutch hydraulic fluid", "clutch hydraulic reservoir", "clutch hydraulic pump",
        "ignition switch", "ignition key", "ignition lock cylinder", "ignition control module",
        "ignition timing", "ignition advance", "ignition coil pack",
        "exhaust manifold", "exhaust pipe", "exhaust muffler", "exhaust catalytic converter",
        "exhaust oxygen sensor", "exhaust temperature sensor", "exhaust backpressure sensor",
        "airbag sensor", "airbag control module", "airbag inflator", "airbag deployment system",
        "seat belt pretensioner", "seat belt retractor", "seat belt buckle",
        "seat belt anchor", "seat belt webbing", "seat belt adjuster", "seat belt warning system",


    ],
    "Energy": [
        "battery", "solar cell", "fuel cell", "photovoltaic", "turbine",
        "energy storage", "renewable energy", "power generation",
        "energy conversion", "energy efficiency", "energy management", "energy harvesting",
        "energy distribution", "energy transmission", "energy grid", "energy infrastructure",
        "solar panel", "wind turbine", "hydroelectric power", "geothermal energy", "biomass energy",
        "nuclear energy", "fossil fuel", "coal power", "natural gas power", "oil power", "hydrogen energy", 
        "energy policy", "energy regulation", "energy market", "energy economics", "energy finance", 
        "energy investment", "energy technology", "energy innovation", "energy research", "energy development", 
        "energy application", "energy integration", "energy interoperability", "energy performance", 
        "energy optimization", "energy monitoring", "energy control", "energy automation", 
        "energy communication", "energy networking", "energy security", "energy resilience", 
        "energy sustainability", "energy environmental impact", "energy social impact", "energy economic impact",
        "hydro power", "wind power", "solar power", "geothermal power", "biomass power",
        "nuclear power", "fossil fuel power", "coal power", "natural gas power", "oil power", 
        "hydrogen power", "energy storage system", "energy storage technology", 
        "wind turbine blade", "wind turbine generator", "wind turbine tower", "wind turbine control system",
        "solar pannel array", "solar panel inverter", "solar panel tracking system", "solar panel mounting system",
        "fuel cell stack", "fuel cell system", "fuel cell membrane", "fuel cell catalyst", 
        "fuel cell bipolar plate", "fuel cell gas diffusion layer"

    ],
    "Chemicals/Materials": [
        "polymer", "composite material", "coating", "catalyst",
        "chemical composition", "alloy", "resin", "nanomaterial", "ion",
        "molecule", "chemical reaction", "chemical process", "chemical synthesis",
        "chemical engineering", "chemical analysis", "chemical testing", "chemical characterization",
        "ploymer synthesis", "polymerization", "polymer processing", "polymer characterization",
        "catalyst synthesis", "catalyst preparation", "catalyst characterization", "catalyst testing",
        "coating formulation", "coating application", "coating characterization", "coating testing",
        "composite material fabrication", "composite material characterization", "composite material testing"
    ],
    "Agriculture": [
        "crop", "fertilizer", "pesticide", "seed", "plant breeding",
        "agricultural", "irrigation", "herbicide", "agronomy", "soil", "livestock",
        "disease resistance", "genetic modification", "plant growth", "harvest", 
        "agriculture technology", "precision agriculture", "sustainable agriculture", "agricultural machinery", "agricultural equipment",
        "agricultural engineering", "agricultural research", "agricultural development",
        "irrigation system", "irrigation technology", "irrigation management", "irrigation scheduling",
        "seed technology", "seed treatment", "seed germination", "seedling", "seedling growth",
        "germination", "plant physiology", "plant pathology", "plant genetics", "plant biotechnology",
        "plant tissue culture", "plant molecular biology", "plant biochemistry", "plant ecology",
        "tissue culture", "plantation", "crop rotation", "crop yield", "crop protection", "crop management", "crop monitoring",
        "crop modeling", "crop simulation", "crop forecasting", "crop prediction", "crop optimization", "crop improvement", "crop breeding", "crop genetics", "crop biotechnology",
        "crop physiology", "crop pathology", "crop biochemistry", "crop ecology", "herbology", "herb",
        "meat", "poultry", "dairy", "aquaculture", "fisheries", "horticulture", "floriculture", 
        "forestry", "agroforestry", "agronomic practices", "agronomic research", "agronomic development",
        "soil fertility", "soil management", "soil conservation", "soil erosion", "soil testing", "soil analysis",
        "erosion", "fertilization", "composting", "mulching", "cover cropping", "crop rotation", "intercropping",
        "agroecology", "agrochemicals", "agroindustry", "agroprocessing", "agroforestry", "agroecosystem", "agroclimatology", "agrometeorology",
        "agrohydrology", "agrogeology", "agroecological zoning", "agroecological modeling", "agroecological assessment", "agroecological monitoring", "agroecological management",
        "agroecological restoration", "agroecological conservation"
    ],
}

def classify_industry(abstract, keyword_map):
    if not isinstance(abstract, str):
        return "Unclassified"
    text = abstract.lower()
    scores = {}
    for industry, keywords in keyword_map.items():
        count = sum(1 for kw in keywords if re.search(r'\b' + re.escape(kw) + r'\b', text))
        if count > 0:
            scores[industry] = count
    if not scores:
        return "Other"
    return max(scores, key=scores.get)

df['industry'] = df['abstract'].apply(lambda x: classify_industry(x, industry_keywords))

In [30]:
df.tail(10)

,patent_id,title,status,expiry_date,filing_date,abstract,inventor_name,raw_xml,publication_date,industry
623,823961,Siloxane derivatives of amino acids having sur...,Under Examination,2040-08-11,2020-08-11,The present disclosure provides siloxane deriv...,ADVANSIX RESINS & CHEMICALS LLC,"<PatentInformationResponse xmlns=""http://www.i...",None,Pharmaceuticals/Biotech
624,786145,A water heating system and an intake and exhau...,Filed,2042-03-11,2022-03-11,An intake and exhaust system for supplying air...,Pump & Electrical Engineering Services Pty Ltd,"<PatentInformationResponse xmlns=""http://www.i...",None,Automotive
625,786146,IMPROVED LIPID NANOPARTICLES FOR DELIVERY OF N...,Under Examination,2040-08-14,2020-08-14,Lipid nanoparticles having improved properties...,"ACUITAS THERAPEUTICS, INC.","<PatentInformationResponse xmlns=""http://www.i...",None,Pharmaceuticals/Biotech
626,814481,Improved lipid nanoparticles for delivery of n...,Under Examination,2040-08-14,2020-08-14,Lipid nanoparticles having improved properties...,"ACUITAS THERAPEUTICS, INC.","<PatentInformationResponse xmlns=""http://www.i...",None,Pharmaceuticals/Biotech
627,786147,SYSTEMS AND METHODS FOR CONTROLLING LASER PULSING,Under Examination,2040-09-25,2020-09-25,Techniques are provided for controlling an out...,"Boston Scientific Scimed, Inc.","<PatentInformationResponse xmlns=""http://www.i...",None,Medical Devices
628,786148,KV1.3 BLOCKERS,Granted,2040-09-18,2020-09-18,The present invention provides novel blockers ...,ZEALAND PHARMA A/S,"<PatentInformationResponse xmlns=""http://www.i...",2025-09-26,Chemicals/Materials
629,786149,SOLID FOOD AND SOLID MILK,Under Examination,2040-09-03,2020-09-03,Provided are a solid food and solid milk havin...,"Meiji Co., Ltd.","<PatentInformationResponse xmlns=""http://www.i...",None,Medical Devices
630,786150,SOLID FOOD AND SOLID MILK,Under Examination,2040-09-03,2020-09-03,Provided are a solid food that enables the man...,"Meiji Co., Ltd.","<PatentInformationResponse xmlns=""http://www.i...",None,Other
631,786300,"FUSED PYRIDONE COMPOUND, AND PREPARATION METHO...",Filed,2040-09-21,2020-09-21,Disclosed in the present invention are a fused...,"SHANGHAI JEYOU PHARMACEUTICAL CO., LTD","<PatentInformationResponse xmlns=""http://www.i...",None,Pharmaceuticals/Biotech
632,786101,ADENO-ASSOCIATED VIRUS VECTOR DELIVERY OF ALPH...,Under Examination,2040-08-21,2020-08-21,Described herein are methods of treating muscu...,RESEARCH INSTITUTE AT NATIONWIDE CHILDREN'S HO...,"<PatentInformationResponse xmlns=""http://www.i...",None,Other
